In [1]:
import pandas as pd
from IPython.display import display
from src.research_config import ResearchConfig
from src.pair_eligibility import filter_antipersistent_pairs
from src.pair_eligibility import compute_structural_t70
from src.pair_eligibility import select_top_pairs_by_structural_t70


# 03 Pair Eligibility

Filter stable anti-persistent fits, estimate structural convergence horizons and select up to 40 pairs. A smaller valid population is reported without loosening thresholds.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
cfg = ResearchConfig().validate()


## 2. Anti persistent and stable fits

Require 0 < H < 0.5, positive sigma/variance and daily Euler stability 0 < kappa < 2.


In [2]:
fou = pd.read_parquet("fractional_ou_parameters.parquet")
pool = filter_antipersistent_pairs(fou)
print(f"{len(pool)} of {len(fou)} fits pass model eligibility")
if pool.empty:
    raise ValueError("No stable anti-persistent fits.")
display(pool.head())


67 of 98 fits pass model eligibility


,pair,dependent,independent,mu,kappa,sigma,hurst,variance,drift_half_life,n_obs
0,A-MTD,A,MTD,7.955231e-15,0.019417,0.012516,0.482774,0.003472,35.697433,1759
1,AAPL-MPWR,AAPL,MPWR,8.602051e-15,0.008272,0.019680,0.411488,0.009393,83.792752,1759
2,ABT-STE,ABT,STE,-6.498497e-15,0.021089,0.014017,0.487555,0.004188,32.868084,1759
3,ABT-ZTS,ABT,ZTS,-7.119061e-15,0.021357,0.013038,0.492385,0.003729,32.455613,1759
4,ADBE-DPZ,ADBE,DPZ,3.587857e-14,0.011598,0.027156,0.436862,0.017258,59.762358,1759


## 3. Structural convergence horizons

This is the expensive formation simulation. Its target, path count and seed are the settings saved in Module 01.


In [3]:
structural = compute_structural_t70(
    pool,
    starting_z=cfg.entry_z,
    target_probability=cfg.target_probability,
    max_horizon_days=cfg.structural_horizon,
    n_paths=cfg.n_paths,
    seed=cfg.seed,
)
structural.to_parquet("structural_results.parquet")
display(structural[["pair", "structural_t70", "structural_probability_max"]].head(15))


,pair,structural_t70,structural_probability_max
0,A-MTD,76.0,0.9894
1,AAPL-MPWR,141.0,0.9052
2,ABT-STE,70.0,0.9922
3,ABT-ZTS,70.0,0.9922
4,ADBE-DPZ,111.0,0.9572
5,ADI-CDNS,NaN,0.5280
6,ADI-KEYS,NaN,0.6068
7,ADI-TXN,238.0,0.7256
8,ADP-MSI,95.0,0.9788
9,ADP-RSG,92.0,0.9774


## 4. Rank and save the portfolio

The full finite-horizon pool is saved separately for matched placebo sampling.


In [4]:
eligible_pool = structural.loc[structural.structural_t70.notna()].copy()
top_pairs = select_top_pairs_by_structural_t70(eligible_pool, cfg.top_n)
eligible_pool.to_parquet("eligible_pool.parquet")
top_pairs.to_parquet("eligible_pairs.parquet")
if top_pairs.empty:
    raise ValueError("No finite structural horizons.")
top_pairs.structural_t70.describe().to_frame("horizon").to_parquet(
    "selected_horizon_summary.parquet"
)
print(f"{len(top_pairs)} selected pairs from {len(eligible_pool)} eligible pairs")
display(top_pairs[["pair", "hurst", "structural_t70", "structural_probability_max"]])


40 selected pairs from 57 eligible pairs


,pair,hurst,structural_t70,structural_probability_max
0,MLM-VMC,0.480033,45.0,0.9994
1,HD-SHW,0.478030,50.0,0.9992
2,MCO-SPGI,0.477245,59.0,0.9976
3,AXP-HLT,0.437350,66.0,0.9962
4,AEP-ETR,0.452053,66.0,0.9958
5,HSY-PEP,0.456088,67.0,0.9954
6,SPGI-ZTS,0.469025,69.0,0.9942
7,ABT-STE,0.487555,70.0,0.9922
8,ABT-ZTS,0.492385,70.0,0.9922
9,MCO-ZTS,0.470511,72.0,0.9918


## Save module completion

Wait for this confirmation before moving to the next notebook.
